# День 3 — EDA и baseline

## Цель
Научиться **смотреть на данные** до модели (EDA) и строить **baseline** — простое правило, с которым сравниваем любую ML-модель.

## EDA — Exploratory Data Analysis (разведочный анализ)

**EDA** — это этап, когда мы **изучаем датасет глазами и цифрами**, ещё не обучая модель.

Зачем:
- понять, что за данные (размер, типы, пропуски);
- заметить выбросы и ошибки;
- сформулировать гипотезы (*«женщины выживали чаще?»*, *«цена зависит от площади?»*);
- решить, какие признаки брать в модель.

Правило: **сначала EDA, потом модель**. Модель на «грязных» или непонятых данных — мусор на входе, мусор на выходе.

## Первичный осмотр таблицы

Базовый набор pandas — как в week-1 day 2 и week-2 day 7:

| Метод | Что даёт |
|-------|----------|
| `head()` | первые строки — «как выглядит» таблица |
| `info()` | типы колонок, число непустых значений |
| `describe()` | статистика по числовым колонкам (mean, std, min, max) |
| `isnull().sum()` | пропуски по каждой колонке |
| `value_counts()` | частоты категорий (для целевой переменной — баланс классов) |

После осмотра часто делают: заполнение пропусков, удаление лишних колонок, кодирование категорий — но **осмысленно**, а не «на автомате».

## Визуализация

Графики помогают увидеть то, что в таблице неочевидно:

- **гистограмма** (`hist`, `plt.hist`) — распределение числа (возраст, цена);
- **bar chart** — сравнение категорий (сколько выжило / не выжило);
- **корреляции / scatter** — связь между двумя признаками.

На EDA не нужны идеальные графики — нужны **ответы на вопросы**: есть ли перекос, выбросы, дубликаты, странные значения.

## Baseline — базовая линия

**Baseline** — наивное, простое правило предсказания **без ML** (или с минимальной логикой). Это «планка», которую модель **обязана** перепрыгнуть.

Примеры:

| Задача | Baseline |
|--------|----------|
| Классификация | всегда предсказывать **самый частый класс** |
| Регрессия | всегда предсказывать **среднее** (или медиану) y |
| Titanic | «все погибли» → accuracy ≈ 62% (доля класса 0) |

Если твоя модель **хуже или равна** baseline — она бесполезна, сколько бы слоёв ни было.

## Зачем baseline перед моделью

1. **Проверка смысла задачи** — данные вообще предсказуемы?
2. **Честное сравнение** — accuracy 0.85 звучит хорошо, но если baseline уже 0.80, выигрыш маленький.
3. **Быстрый старт** — baseline считается за минуты; сложная модель — только если baseline побит.

Типичный порядок дня:

```
постановка задачи (day 2) → EDA → split → baseline на test → модель
```

**Важно про порядок:**
- **EDA** (info, графики) — смотрим на **весь** датасет, чтобы понять данные.
- **Split** — делим на train/test **до** baseline и модели (как в day 2).
- **Baseline** — `fit` на train, метрика на **test** (или CV). Иначе оценка нечестная.


## Задания

In [ ]:
# Загрузка + EDA: info, describe, пропуски, распределение target

import pandas as pd
from sklearn.datasets import load_iris

dataset = load_iris(as_frame=True)
df = dataset.frame

df.info()

In [ ]:
df.describe() 

In [ ]:
df.isnull().sum() 

In [ ]:
df["target"].value_counts()

In [ ]:
# 3. Три графика → сохранить в week-3/figures/

from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

FIGURES = Path("week-3/figures")
if not FIGURES.exists():
    FIGURES = Path("../figures")  # если запуск из week-3/notebooks/
FIGURES.mkdir(parents=True, exist_ok=True)

dataset = load_iris(as_frame=True)
df = dataset.frame
names = dataset.target_names

# 1) баланс классов (bar)
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["target"].value_counts().sort_index()
ax.bar(names, counts.values)
ax.set_title("Iris: class balance")
ax.set_ylabel("count")
fig.savefig(FIGURES / "01_target_balance.png", dpi=120)
plt.show()

# 2) гистограмма признака
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["petal length (cm)"], bins=15, edgecolor="white")
ax.set_title("Petal length distribution")
ax.set_xlabel("petal length (cm)")
fig.savefig(FIGURES / "02_petal_length_hist.png", dpi=120)
plt.show()

# 3) scatter: два признака, цвет = класс
fig, ax = plt.subplots(figsize=(6, 4))
for t, name in enumerate(names):
    m = df["target"] == t
    ax.scatter(df.loc[m, "petal length (cm)"], df.loc[m, "petal width (cm)"], label=name, alpha=0.8)
ax.set_title("Petal length vs width")
ax.legend()
fig.savefig(FIGURES / "03_petal_scatter.png", dpi=120)
plt.show()

print("Сохранено в:", FIGURES.resolve())


## X/y и train/test split

После EDA — как в day 2: разделяем признаки и цель, затем делаем split **до** baseline.

- `fit` — только на **train**
- метрика baseline — на **test**


In [ ]:
# 4. X, y и train_test_split (те же параметры, что в day 2)

from sklearn.model_selection import train_test_split

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


In [ ]:
# 5. Baseline: fit на train, метрика на test

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

acc = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline accuracy: {acc:.2f}")


## Выводы (5–7 строк)

1. Iris: 150 строк, 4 признака + target; пропусков нет, все признаки числовые (`float64`).
2. Классы **сбалансированы** (по 50 образцов) — дисбаланса нет, stratify при split не критичен, но полезен.
3. `describe()` показывает разный масштаб признаков (лепестки уже, чем чашелистики) — для многих моделей понадобится нормализация.
4. Графики: **petal length** и scatter petal length vs width хорошо разделяют виды; setosa отделён, versicolor и virginica частично пересекаются.
5. Split 120/30 (stratify) — train и test с одинаковой долей классов.
6. Baseline (`most_frequent`) на test: **accuracy = 0.33** — при трёх равных классах это ожидаемо (~1/3 угадываний).
7. Планка низкая: любая модель, использующая признаки (особенно лепестки), должна заметно превзойти 0.33.

---

# Day 3 — EDA and baseline

## Goal
Learn to **look at the data** before modeling (EDA) and build a **baseline** — a simple rule to compare any ML model against.

## EDA — Exploratory Data Analysis

**EDA** is the stage where we **explore the dataset with eyes and numbers** before training a model.

Why:
- understand the data (size, types, missing values);
- spot outliers and errors;
- form hypotheses (*"did women survive more often?"*, *"does price depend on area?"*);
- decide which features to use.

Rule: **EDA first, model second**. A model on dirty or misunderstood data — garbage in, garbage out.

## Initial table inspection

Basic pandas toolkit — as in week-1 day 2 and week-2 day 7:

| Method | What it shows |
|--------|---------------|
| `head()` | first rows — what the table looks like |
| `info()` | column types, non-null counts |
| `describe()` | stats for numeric columns (mean, std, min, max) |
| `isnull().sum()` | missing values per column |
| `value_counts()` | category frequencies (for target — class balance) |

After inspection: often fill missing values, drop useless columns, encode categories — but **deliberately**, not on autopilot.

## Visualization

Plots reveal what tables hide:

- **histogram** (`hist`, `plt.hist`) — distribution of a number (age, price);
- **bar chart** — compare categories (survived vs not);
- **correlation / scatter** — relationship between two features.

EDA plots don't need to be perfect — they need to **answer questions**: skew, outliers, duplicates, odd values.

## Baseline

A **baseline** is a naive, simple prediction rule **without ML** (or with minimal logic). It is the bar a model **must** beat.

Examples:

| Task | Baseline |
|------|----------|
| Classification | always predict the **most frequent class** |
| Regression | always predict the **mean** (or median) of y |
| Titanic | "everyone died" → accuracy ≈ 62% (share of class 0) |

If your model is **worse than or equal to** the baseline — it is useless, no matter how many layers it has.

## Why baseline before a model

1. **Sanity check** — is the task predictable at all?
2. **Honest comparison** — 0.85 accuracy sounds good, but if baseline is already 0.80, the gain is small.
3. **Fast start** — baseline takes minutes; use a complex model only if baseline is beaten.

Typical flow for the day:

```
problem setting (day 2) → EDA → split → baseline on test → model
```

**Order matters:**
- **EDA** (info, plots) — use the **full** dataset to understand the data.
- **Split** — train/test **before** baseline and model (as in day 2).
- **Baseline** — `fit` on train, metric on **test** (or CV). Otherwise the score is not honest.


## Tasks (do by hand)

1. Notebook: `week-3/notebooks/day03_eda_baseline.ipynb`.
2. Run `info()` / `describe()`, check missing values, types, and **target** distribution.
3. Build **3 plots** and save to `week-3/figures/`.
4. Build a **baseline**:
   - classification → `DummyClassifier(strategy="most_frequent")`
   - regression → `DummyRegressor(strategy="mean")`
5. Compute the baseline metric and write **5–7 lines of conclusions**: what matters in the data for the model.

## Conclusions (5–7 lines)

1. Iris: 150 rows, 4 features + target; no missing values, all numeric.
2. Classes are **balanced** (50 each) — no class imbalance.
3. Features have different scales — scaling may help later models.
4. Plots: **petal** features separate species well; setosa is isolated, versicolor/virginica overlap somewhat.
5. Train/test split 120/30 with stratify keeps class proportions.
6. Baseline accuracy on test: **0.33** — expected when always predicting one of three equal classes.
7. Any sensible model using features should beat 0.33 by a large margin.